# Experiment 3 - Market Match-Date + Macro Fill

## Install & Import Library

In [ ]:
# install libraryt
!pip install statsmodels yfinance pandas-datareader requests scikit-learn xgboost lightgbm optuna plotly --quiet

In [ ]:
# import some libarries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import os
import json
import pickle
import random

import requests
import re
import yfinance as yf
from datetime import datetime

from openpyxl import load_workbook

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import xgboost as xgb
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [ ]:
# set reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 42
set_seed(seed)

## Konfigurasi

In [ ]:
# path dataset
PATH_ANTAM = '/kaggle/input/datasets/naufalfz/dataset/harga_emas_antm.csv'
PATH_XAU = '/kaggle/input/datasets/naufalfz/dataset/Data Historis XAU_USD.csv'
PATH_KURS = '/kaggle/input/datasets/naufalfz/dataset/data kurs usd idr.xlsx'
PATH_BI = '/kaggle/input/datasets/naufalfz/dataset/BI-7Day-RR.xlsx'
PATH_INFLASI = '/kaggle/input/datasets/naufalfz/dataset/Data Inflasi.xlsx'

In [ ]:
# config crawling, feature engineering params modeling
START_DATE = '2019-01-01'
END_DATE = '2026-02-20'
TRAIN_RATIO = 0.70
MAX_HORIZON = 7
SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEAS = (0, 0, 0, 0)
N_OPTUNA_TRIALS = 50

## Load & Merge Dataset

In [ ]:
BULAN_ID = {
    'Januari':'01','Februari':'02','Maret':'03','April':'04',
    'Mei':'05','Juni':'06','Juli':'07','Agustus':'08',
    'September':'09','Oktober':'10','November':'11','Desember':'12'
}

def parse_tgl_id(s):
    p = s.strip().split()
    return pd.to_datetime(f"{p[2]}-{BULAN_ID[p[1]]}-{p[0].zfill(2)}")

def parse_periode_id(s):
    p = s.strip().split()
    return pd.to_datetime(f"{p[1]}-{BULAN_ID[p[0]]}-01")

# 1. Harga Emas Antam
df_antam = pd.read_csv(PATH_ANTAM)
df_antam['tanggal'] = pd.to_datetime(df_antam['tanggal'])
df_antam = df_antam[['tanggal','harga']].rename(columns={'harga':'harga_emas_antam_idr'}).set_index('tanggal').sort_index()

# 2. XAU/USD (Investing.com)
df_xau = pd.read_csv(PATH_XAU)
df_xau['Tanggal'] = pd.to_datetime(df_xau['Tanggal'], format='%d/%m/%Y')
df_xau['harga_emas_dunia_usd'] = (df_xau['Terakhir']
    .str.replace('.','',regex=False).str.replace(',','.',regex=False).astype(float))
df_xau = df_xau[['Tanggal','harga_emas_dunia_usd']].set_index('Tanggal').sort_index()

# 3. Kurs USD/IDR (yfinance)
df_kurs = pd.read_excel(PATH_KURS)
df_kurs.drop('Unnamed: 0', axis=1, inplace=True)
df_kurs.columns = ['tanggal','kurs_usdidr']
df_kurs['tanggal'] = pd.to_datetime(df_kurs['tanggal'])
df_kurs = df_kurs[['tanggal','kurs_usdidr']].set_index('tanggal').sort_index()

# 4. BI 7-Day RR Rate (Bank Indonesia)
wb_bi = load_workbook(PATH_BI, read_only=True); ws_bi = wb_bi.active
bi_rows = []
for row in ws_bi.iter_rows(values_only=True):
    if row[0] is not None and isinstance(row[0], int):
        bi_rows.append({'tanggal': parse_tgl_id(row[1]),
                        'bi_7drr_rate': float(row[2].replace('%','').strip())})
df_bi = pd.DataFrame(bi_rows).set_index('tanggal').sort_index()

# 5. Inflasi YoY (Bank Indonesia)
wb_inf = load_workbook(PATH_INFLASI, read_only=True); ws_inf = wb_inf.active
inf_rows = []
for row in ws_inf.iter_rows(values_only=True):
    if row[0] is not None and isinstance(row[0], int):
        inf_rows.append({'tanggal': parse_periode_id(row[1]),
                         'inflasi_yoy_id': float(row[2].replace('%','').strip())})
df_inflasi = pd.DataFrame(inf_rows).set_index('tanggal').sort_index()

In [ ]:
# fitur yang digunakan untuk model boosting
FEATURE_COLS = [
    'log_return',
    'lag_1', 'lag_2', 'lag_3', 'lag_5', 'lag_7',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14',
    'log_xau', 'log_kurs', 'log_xau_lag1', 'log_kurs_lag1',
    'bi_7drr_rate', 'inflasi_yoy_id', 'spread_macro',
    'day_of_week', 'month', 'quarter'
]

# eksogen SARIMAX: XAU/USD & Kurs saja
EXOG_SARIMAX = ['log_xau', 'log_kurs']

print(f'SARIMAX exogenous : {EXOG_SARIMAX}')
print(f'Boosting features : {len(FEATURE_COLS)} kolom')


SARIMAX exogenous : ['log_xau', 'log_kurs']
Boosting features : 20 kolom


In [ ]:
# helper function
def evaluate_forecast(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred)
    }

tscv = TimeSeriesSplit(n_splits=5)


In [ ]:
# Merge tanpa ffill/bfill kecuali inflasi dan birate
full_idx_e3 = pd.date_range(start=START_DATE, end=END_DATE, freq='D')

df_e3 = (
    df_antam
    .join(df_xau, how='inner')
    .join(df_kurs, how='inner')
)

df_e3 = df_e3.loc[START_DATE:END_DATE].copy()

# untuk macro reindex ke daily lalu fill
macro_e3 = pd.DataFrame(index=full_idx_e3)
macro_e3.index.name = 'tanggal'

macro_e3 = macro_e3.join(df_bi.reindex(full_idx_e3))
macro_e3 = macro_e3.join(df_inflasi.reindex(full_idx_e3))

# Sesuai izin client: hanya BI rate dan inflasi yang boleh fill
macro_e3 = macro_e3.ffill().bfill()

df_e3 = df_e3.join(macro_e3, how='left')
df_e3 = df_e3.dropna().copy()

print(f'Shape  : {df_e3.shape}')
print(f'Periode: {df_e3.index[0].date()} s/d {df_e3.index[-1].date()}')
print(f'Missing:\n{df_e3.isnull().sum()}')

Shape  : (1804, 5)
Periode: 2019-01-02 s/d 2026-02-19
Missing:
harga_emas_antam_idr    0
harga_emas_dunia_usd    0
kurs_usdidr             0
bi_7drr_rate            0
inflasi_yoy_id          0
dtype: int64


## Uji Stasioneritas (ADF Test)

In [ ]:
# function adf test (cek stasioner)
def adf_test(series, name):
    res = adfuller(series.dropna(), autolag='AIC')
    status = 'Stasioner' if res[1] < 0.05 else 'Tidak Stasioner'
    print(f'{name:<35} | ADF: {res[0]:>8.4f} | p: {res[1]:.4f} | {status}')

print('Level:')
adf_test(df_e3['harga_emas_antam_idr'], 'Harga Emas Antam (level)')
adf_test(df_e3['harga_emas_dunia_usd'], 'Harga Emas Dunia (level)')
adf_test(df_e3['kurs_usdidr'], 'Kurs USD/IDR (level)')

df_e3['log_harga'] = np.log(df_e3['harga_emas_antam_idr'])
df_e3['log_xau'] = np.log(df_e3['harga_emas_dunia_usd'])
df_e3['log_kurs'] = np.log(df_e3['kurs_usdidr'])

print('\nLog')
adf_test(df_e3['log_harga'], 'log(Harga Emas Antam)')
adf_test(df_e3['log_xau'], 'log(Harga Emas Dunia)')
adf_test(df_e3['log_kurs'], 'log(Kurs USD/IDR)')

print('\nLog First Difference')
adf_test(df_e3['log_harga'].diff().dropna(), 'log(Harga Emas Antam)')
adf_test(df_e3['log_xau'].diff().dropna(), 'log(Harga Emas Dunia)')
adf_test(df_e3['log_kurs'].diff().dropna(), 'log(Kurs USD/IDR)')

Level:
Harga Emas Antam (level)            | ADF:   4.5307 | p: 1.0000 | Tidak Stasioner
Harga Emas Dunia (level)            | ADF:   4.6099 | p: 1.0000 | Tidak Stasioner
Kurs USD/IDR (level)                | ADF:  -1.5470 | p: 0.5101 | Tidak Stasioner

Log
log(Harga Emas Antam)               | ADF:   2.2615 | p: 0.9989 | Tidak Stasioner
log(Harga Emas Dunia)               | ADF:   1.6517 | p: 0.9980 | Tidak Stasioner
log(Kurs USD/IDR)                   | ADF:  -1.6063 | p: 0.4804 | Tidak Stasioner

Log First Difference
log(Harga Emas Antam)               | ADF: -31.1958 | p: 0.0000 | Stasioner
log(Harga Emas Dunia)               | ADF: -31.9028 | p: 0.0000 | Stasioner
log(Kurs USD/IDR)                   | ADF:  -8.8367 | p: 0.0000 | Stasioner


## Feature Engineering


In [ ]:
# feature engineering ttp jg
df_e3['log_harga'] = np.log(df_e3['harga_emas_antam_idr'])
df_e3['log_xau'] = np.log(df_e3['harga_emas_dunia_usd'])
df_e3['log_kurs'] = np.log(df_e3['kurs_usdidr'])

# Log return
df_e3['log_return'] = df_e3['log_harga'].diff()

# Lag target
for lag in [1, 2, 3, 5, 7]:
    df_e3[f'lag_{lag}'] = df_e3['log_harga'].shift(lag)

# Rolling statistics berbasis t-1
df_e3['rolling_mean_7'] = df_e3['log_harga'].shift(1).rolling(7).mean()
df_e3['rolling_std_7'] = df_e3['log_harga'].shift(1).rolling(7).std()
df_e3['rolling_mean_14'] = df_e3['log_harga'].shift(1).rolling(14).mean()
df_e3['rolling_std_14'] = df_e3['log_harga'].shift(1).rolling(14).std()

# Calendar features
df_e3['day_of_week'] = df_e3.index.dayofweek
df_e3['month'] = df_e3.index.month
df_e3['quarter'] = df_e3.index.quarter

# Macro spread
df_e3['spread_macro'] = df_e3['bi_7drr_rate'] - df_e3['inflasi_yoy_id']

# Lag eksogen
df_e3['log_xau_lag1'] = df_e3['log_xau'].shift(1)
df_e3['log_kurs_lag1'] = df_e3['log_kurs'].shift(1)

# Drop NaN akibat diff, lag, rolling
df_e3 = df_e3.dropna().copy()

missing_cols_e3 = [c for c in FEATURE_COLS if c not in df_e3.columns]

print(f'Shape after feature engineering: {df_e3.shape}')
print(f'Jumlah fitur boosting: {len(FEATURE_COLS)}')

Shape after feature engineering: (1790, 24)
Jumlah fitur boosting: 20


In [ ]:
# Bersihkan nilai inf setelah feature engineering
df_e3 = df_e3.replace([np.inf, -np.inf], np.nan)
df_e3 = df_e3.dropna().copy()

# Safety check
num_cols_e3 = df_e3.select_dtypes(include=[np.number]).columns
non_finite_count = (~np.isfinite(df_e3[num_cols_e3])).sum().sum()

print(f'Non-finite values after cleaning: {non_finite_count}')
print(f'Shape after finite cleaning: {df_e3.shape}')

if non_finite_count > 0:
    raise ValueError('Masih ada nilai inf/-inf di df_e3.')

In [ ]:
# train test split
split_idx_e3 = int(len(df_e3) * TRAIN_RATIO)

train_e3 = df_e3.iloc[:split_idx_e3].copy()
test_e3  = df_e3.iloc[split_idx_e3:].copy()

print(f'Train: {len(train_e3)}')
print(f'Test : {len(test_e3)}')

missing_exog_e3 = [c for c in EXOG_SARIMAX if c not in df_e3.columns]
missing_feat_e3 = [c for c in FEATURE_COLS if c not in df_e3.columns]

Train: 1253
Test : 537


In [ ]:
# SARIMAX baseline + optuna tuning
sarimax_e3 = SARIMAX(
    train_e3['log_harga'],
    exog=train_e3[EXOG_SARIMAX],
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SEAS,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False, maxiter=200)

train_e3['residual'] = train_e3['log_harga'].values - sarimax_e3.fittedvalues.values

# Bersihkan residual yang inf / terlalu besar
train_e3 = train_e3.replace([np.inf, -np.inf], np.nan)
train_e3 = train_e3.dropna(subset=['residual'] + FEATURE_COLS + EXOG_SARIMAX).copy()

X_train_e3 = train_e3[FEATURE_COLS].values
y_train_e3 = train_e3['residual'].values

finite_mask_e3 = (
    np.isfinite(X_train_e3).all(axis=1)
    & np.isfinite(y_train_e3)
)

X_train_e3 = X_train_e3[finite_mask_e3]
y_train_e3 = y_train_e3[finite_mask_e3]

print(f'X_train_e3 shape after finite filter: {X_train_e3.shape}')
print(f'y_train_e3 shape after finite filter: {y_train_e3.shape}')

if len(y_train_e3) == 0:
    raise ValueError('y_train_e3 kosong setelah finite filter.')


def objective_xgb_e3(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'random_state': seed,
        'n_jobs': -1,
    }

    scores = []

    for tr_idx, va_idx in tscv.split(X_train_e3):
        m = xgb.XGBRegressor(**params)
        m.fit(
            X_train_e3[tr_idx],
            y_train_e3[tr_idx],
            eval_set=[(X_train_e3[va_idx], y_train_e3[va_idx])],
            verbose=False
        )

        pred_va = m.predict(X_train_e3[va_idx])
        scores.append(mean_squared_error(y_train_e3[va_idx], pred_va))

    return np.mean(scores)


def objective_lgb_e3(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'random_state': seed,
        'n_jobs': -1,
        'verbose': -1,
    }

    scores = []

    for tr_idx, va_idx in tscv.split(X_train_e3):
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_train_e3[tr_idx],
            y_train_e3[tr_idx],
            eval_set=[(X_train_e3[va_idx], y_train_e3[va_idx])],
            callbacks=[
                lgb.early_stopping(20, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )

        pred_va = m.predict(X_train_e3[va_idx])
        scores.append(mean_squared_error(y_train_e3[va_idx], pred_va))

    return np.mean(scores)


study_xgb_e3 = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=seed)
)

study_xgb_e3.optimize(
    objective_xgb_e3,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True
)

best_xgb_params_e3 = {
    **study_xgb_e3.best_params,
    'random_state': seed,
    'n_jobs': -1,
}


study_lgb_e3 = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=seed)
)

study_lgb_e3.optimize(
    objective_lgb_e3,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True
)

best_lgb_params_e3 = {
    **study_lgb_e3.best_params,
    'random_state': seed,
    'n_jobs': -1,
    'verbose': -1,
}

print(f'Best XGB Exp3: {best_xgb_params_e3}')
print(f'Best LGB Exp3: {best_lgb_params_e3}')

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Best XGB Exp3: {'n_estimators': 499, 'learning_rate': 0.07161350133685537, 'max_depth': 5, 'subsample': 0.7525025590597331, 'colsample_bytree': 0.8293511135712811, 'reg_alpha': 9.939903058090557, 'reg_lambda': 0.004120091712651, 'random_state': 42, 'n_jobs': -1}
Best LGB Exp3: {'n_estimators': 332, 'learning_rate': 0.29699989529527066, 'max_depth': 5, 'num_leaves': 68, 'subsample': 0.8009138391657643, 'colsample_bytree': 0.9310823788613438, 'reg_alpha': 0.0008790566924557089, 'reg_lambda': 0.01345125320371781, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


In [ ]:
# Direct Multi-Step Hybrid Forecast
all_results_e3 = []
metrics_rows_e3 = []

for h in range(1, MAX_HORIZON + 1):
    print(f'(Exp3-MarketMatch-MacroFill) Horizon H+{h}')

    train_pairs = []

    for t in range(len(train_e3) - h):
        history_y = train_e3['log_harga'].iloc[:t + 1]
        history_exog = train_e3[EXOG_SARIMAX].iloc[:t + 1]
        future_exog = train_e3[EXOG_SARIMAX].iloc[t + 1:t + 1 + h]

        if len(future_exog) < h:
            continue

        try:
            m_hist = SARIMAX(
                history_y,
                exog=history_exog,
                order=SARIMAX_ORDER,
                seasonal_order=SARIMAX_SEAS,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False, maxiter=200)

            fc_h = m_hist.forecast(steps=h, exog=future_exog).iloc[-1]
            residual_h = train_e3['log_harga'].iloc[t + h] - fc_h

            train_pairs.append(
                (
                    train_e3[FEATURE_COLS].iloc[t].values,
                    residual_h
                )
            )

        except Exception:
            continue

    if len(train_pairs) == 0:
        print(f'H+{h} dilewati karena train_pairs kosong.')
        continue

    X_tr = np.array([p[0] for p in train_pairs])
    y_tr = np.array([p[1] for p in train_pairs])

    xgb_h_e3 = xgb.XGBRegressor(**best_xgb_params_e3)
    xgb_h_e3.fit(X_tr, y_tr)

    lgb_h_e3 = lgb.LGBMRegressor(**best_lgb_params_e3)
    lgb_h_e3.fit(X_tr, y_tr)

    history_wf = train_e3.copy()

    sarimax_preds_log = []
    actual_vals = []
    feat_rows = []

    for i in range(len(test_e3) - h + 1):
        exog_future = test_e3[EXOG_SARIMAX].iloc[i:i + h]

        if len(exog_future) < h:
            break

        try:
            m = SARIMAX(
                history_wf['log_harga'],
                exog=history_wf[EXOG_SARIMAX],
                order=SARIMAX_ORDER,
                seasonal_order=SARIMAX_SEAS,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False, maxiter=200)

            fc = m.forecast(steps=h, exog=exog_future)

            sarimax_preds_log.append(fc.iloc[-1])
            feat_rows.append(test_e3[FEATURE_COLS].iloc[i].values)
            actual_vals.append(test_e3['harga_emas_antam_idr'].iloc[i + h - 1])

            history_wf = pd.concat([history_wf, test_e3.iloc[[i]]])

        except Exception:
            continue

    if len(sarimax_preds_log) == 0:
        print(f'H+{h} dilewati karena hasil forecast kosong.')
        continue

    sarimax_preds_log = np.array(sarimax_preds_log)
    X_feat = np.array(feat_rows)
    actual_arr = np.array(actual_vals)

    xgb_resid_pred = xgb_h_e3.predict(X_feat)
    lgb_resid_pred = lgb_h_e3.predict(X_feat)

    log_pred_s = sarimax_preds_log
    log_pred_xgb = sarimax_preds_log + xgb_resid_pred
    log_pred_lgb = sarimax_preds_log + lgb_resid_pred

    finite_mask = (
        np.isfinite(actual_arr)
        & np.isfinite(log_pred_s)
        & np.isfinite(log_pred_xgb)
        & np.isfinite(log_pred_lgb)
    )

    actual_arr = actual_arr[finite_mask]
    log_pred_s = log_pred_s[finite_mask]
    log_pred_xgb = log_pred_xgb[finite_mask]
    log_pred_lgb = log_pred_lgb[finite_mask]

    if len(actual_arr) == 0:
        print(f'H+{h} dilewati karena semua prediksi non-finite.')
        continue

    log_min_e3 = df_e3['log_harga'].min() - 1.0
    log_max_e3 = df_e3['log_harga'].max() + 1.0

    log_pred_s = np.clip(log_pred_s, log_min_e3, log_max_e3)
    log_pred_xgb = np.clip(log_pred_xgb, log_min_e3, log_max_e3)
    log_pred_lgb = np.clip(log_pred_lgb, log_min_e3, log_max_e3)

    pred_s_e3 = np.exp(log_pred_s)
    pred_xgb_e3 = np.exp(log_pred_xgb)
    pred_lgb_e3 = np.exp(log_pred_lgb)

    finite_pred_mask = (
        np.isfinite(actual_arr)
        & np.isfinite(pred_s_e3)
        & np.isfinite(pred_xgb_e3)
        & np.isfinite(pred_lgb_e3)
    )

    actual_arr = actual_arr[finite_pred_mask]
    pred_s_e3 = pred_s_e3[finite_pred_mask]
    pred_xgb_e3 = pred_xgb_e3[finite_pred_mask]
    pred_lgb_e3 = pred_lgb_e3[finite_pred_mask]

    if len(actual_arr) == 0:
        print(f'H+{h} dilewati karena prediksi tidak valid setelah exp.')
        continue

    res_s_e3 = evaluate_forecast(actual_arr, pred_s_e3)
    res_xgb_e3 = evaluate_forecast(actual_arr, pred_xgb_e3)
    res_lgb_e3 = evaluate_forecast(actual_arr, pred_lgb_e3)

    all_results_e3.append({
        'horizon': h,
        'actual': actual_arr,
        'sarimax_preds': pred_s_e3,
        'xgb_preds': pred_xgb_e3,
        'lgb_preds': pred_lgb_e3,
        'sarimax': res_s_e3,
        'hybrid_xgb': res_xgb_e3,
        'hybrid_lgb': res_lgb_e3,
    })

    metrics_rows_e3.extend([
        {
            'horizon': f'H+{h}',
            'model': 'SARIMAX (market-match macro-fill)',
            'MAE': res_s_e3['mae'],
            'RMSE': res_s_e3['rmse'],
            'R2': res_s_e3['r2']
        },
        {
            'horizon': f'H+{h}',
            'model': 'Hybrid XGB (market-match macro-fill)',
            'MAE': res_xgb_e3['mae'],
            'RMSE': res_xgb_e3['rmse'],
            'R2': res_xgb_e3['r2']
        },
        {
            'horizon': f'H+{h}',
            'model': 'Hybrid LGB (market-match macro-fill)',
            'MAE': res_lgb_e3['mae'],
            'RMSE': res_lgb_e3['rmse'],
            'R2': res_lgb_e3['r2']
        },
    ])

    print(
        f"H+{h} | "
        f"SARIMAX MAE:{res_s_e3['mae']:>10,.0f} R2:{res_s_e3['r2']:>6.3f} | "
        f"XGB MAE:{res_xgb_e3['mae']:>10,.0f} R2:{res_xgb_e3['r2']:>6.3f} | "
        f"LGB MAE:{res_lgb_e3['mae']:>10,.0f} R2:{res_lgb_e3['r2']:>6.3f}"
    )

## Export Metrics


In [ ]:
# export metrics Experiment 3
metrics_exp3 = pd.DataFrame(metrics_rows_e3)
metrics_exp3 = metrics_exp3.sort_values(['horizon', 'model']).reset_index(drop=True)
metrics_exp3.to_csv('metrics_exp3_market_matchdate_macro_fill.csv', index=False)
metrics_exp3
